# Deutschlandticket Adoption Analysis - Johnson & Johnson

## 1. Architecture & Data Generation Note
To ensure clean code and modularity, the heavy lifting of data generation and spatial routing has been separated into dedicated Python scripts, rather than cluttering this analytical notebook.

* **`generate_data.py`**: Uses statistical sampling (`numpy.random`) within real GeoJSON district boundaries of Hamburg to generate 2,000 synthetic employee locations. It assigns realistic car-ownership probabilities and weights the distribution by actual district population density.
* **`compute_routes`**: Utilizes `r5py` (a Java-based routing engine) and public OpenStreetMap/GTFS data to calculate highly accurate door-to-door public transport commute times for all synthetic employees to the J&J Norderstedt office.

This notebook focuses entirely on the **analysis, scoring heuristics, and visualization** of the generated `commute_results.csv` and `employees.csv` datasets.

## 2. Preprocessing

In [ ]:
import pandas as pd
import folium

# Load the generated datasets
employees = pd.read_csv("data/employees.csv")
results = pd.read_csv("data/commute_results.csv")

# Merge on employee ID
df = employees.merge(results, left_on="employee_id", right_on="employee_id", how="left")

# Calculate the mean travel time and switch missing values (NaNs)
mean_time = df['travel_time_min'].mean()
df['travel_time_min'] = df['travel_time_min'].fillna(mean_time)

print(f"Total employees: {len(df)}")
print(f"Missing values filled with mean commute time: {mean_time:.1f} minutes")
df.head()

Total employees: 2000
Missing values filled with mean commute time: 81.3 minutes


,employee_id,stadtteil,lat,lon,hat_auto,travel_time_min
0,EMP0001,Poppenbüttel,53.662518,10.069617,True,61.0
1,EMP0002,Norderstedt,53.708102,10.048249,True,71.0
2,EMP0003,Norderstedt,53.672568,9.959442,False,63.0
3,EMP0004,Langenhorn,53.664344,9.992296,True,40.0
4,EMP0005,Bramfeld,53.605512,10.068412,True,81.0


## 2. Commute-Time Grouping & Adoption Scoring

To estimate the adoption potential for the Deutschlandticket, we apply a business heuristic based on two main factors:
1. **Commute Time:** Shorter public transport commutes lead to a higher likelihood of adoption. We categorize times into <30, 30-45, 45-60, and 60+ minutes.
2. **Car Ownership:** Instead of calculating car routing, we use the generated `hat_auto` variable. Employees who own a car experience a higher convenience baseline. Therefore, car ownership penalizes the base probability of adopting the public transit ticket by a flat 25%.



Since no historical HR data on public transport usage was provided, this model uses a **business heuristic** (an educated estimate) to calculate the likelihood of an employee adopting the Deutschlandticket. The scoring is based on two main principles:

**1. The Commute-Time Curve (Diminishing Returns)**
The likelihood of using public transport drops exponentially as travel time increases.
* **< 30 Min (Score: 0.90):** High convenience. Public transport is highly competitive with driving.
* **30-45 Min (Score: 0.70):** Still attractive, but convenience factors begin to weigh in.
* **45-60 Min (Score: 0.40):** The tipping point. Cars are usually significantly faster for these distances.
* **60+ Min (Score: 0.10):** The pain threshold. Usually only utilized by employees without alternative transport options.

**2. The Car Ownership Penalty (Sunk Costs Proxy)**
Instead of calculating complex multimodal car routes, the `hat_auto` variable acts as a proxy for convenience. Employees who own a car have already paid the "sunk costs" (insurance, depreciation). This creates a psychological barrier to using public transit, which we model as a flat **25% penalty** on their base adoption score.

In [54]:
# Commute-Time Grouping
bins = [0, 30, 45, 60, 200, 1000]
labels = ["<30 Min", "30-45 Min", "45-60 Min", "60+ Min", "No Connection"]
df['time_bucket'] = pd.cut(df['travel_time_min'], bins=bins, labels=labels)


def calculate_adoption(row):
    # Basis-Wahrscheinlichkeit nach Pendelzeit
    if row['time_bucket'] == "<30 Min": score = 0.90
    elif row['time_bucket'] == "30-45 Min": score = 0.70
    elif row['time_bucket'] == "45-60 Min": score = 0.40
    elif row['time_bucket'] == "60+ Min": score = 0.10
   

    # penalty for having a car
    if row['hat_auto'] and score > 0:
        score -= 0.25
        
    return max(0, score) # to ensure only positiv scores 

df['adoption_potential'] = df.apply(calculate_adoption, axis=1)

# flag for the map
df['high_potential'] = df['adoption_potential'] >= 0.7

df[['employee_id', 'hat_auto','stadtteil', 'travel_time_min', 'time_bucket', 'adoption_potential']].head()

,employee_id,hat_auto,stadtteil,travel_time_min,time_bucket,adoption_potential
0,EMP0001,True,Poppenbüttel,61.0,60+ Min,0.00
1,EMP0002,True,Norderstedt,71.0,60+ Min,0.00
2,EMP0003,False,Norderstedt,63.0,60+ Min,0.10
3,EMP0004,True,Langenhorn,40.0,30-45 Min,0.45
4,EMP0005,True,Bramfeld,81.0,60+ Min,0.00


## 3. Summary Findings & Expected Deliverables

The following section generates the final summary requested by the assignment. It calculates the exact distribution of commute times, the overall estimated adoption rate for the Deutschlandticket, and identifies the best and worst connected districts around the J&J Norderstedt office.

In [55]:
# 1. Commute Time Distribution
print("1. Commute Time Distribution:")
bucket_percentages = df['time_bucket'].value_counts(normalize=True) * 100
for bucket, pct in bucket_percentages.items():
    print(f"   - {bucket}: {pct:.1f}%")

# 2. Estimated Deutschlandticket Adoption Potential
average_adoption = df['adoption_potential'].mean() * 100
print(f"\n2. Estimated Deutschlandticket Adoption Potential: {average_adoption:.1f}%")

# 3. Regional Public Transport Connectivity
print("\n3. Regional Public Transport Connectivity:")
# Calculate average commute time per district (excluding 999 penalty times if any still exist)
valid_routes = df[df['travel_time_min'] < 999]
district_times = valid_routes.groupby('stadtteil')['travel_time_min'].mean().sort_values()

print(f"   - Strongest Areas (Shortest commutes): {', '.join(district_times.head(5).index)}")
print(f"   - Weakest Areas (Longest commutes): {', '.join(district_times.tail(5).index)}")

1. Commute Time Distribution:
   - 60+ Min: 91.5%
   - 45-60 Min: 5.3%
   - 30-45 Min: 2.5%
   - <30 Min: 0.5%
   - No Connection: 0.0%

2. Estimated Deutschlandticket Adoption Potential: 5.4%

3. Regional Public Transport Connectivity:
   - Strongest Areas (Shortest commutes): Langenhorn, Hummelsbüttel, Poppenbüttel, Alsterdorf, Ohlsdorf
   - Weakest Areas (Longest commutes): Altona-Altstadt, Lurup, Ottensen, Osdorf, Bahrenfeld


## 4. Interactive map


In [56]:
# Office
jj_lat, jj_lon = 53.6866, 9.9934

# HVV Stations
hvv_stations = {
    "U1 Garstedt": (53.684, 9.986),
    "U1 Richtweg": (53.693, 9.992),
    "U1 Norderstedt Mitte": (53.702, 9.993)
}

m = folium.Map(location=[jj_lat, jj_lon], zoom_start=11, tiles="CartoDB Positron")

folium.Marker(
    [jj_lat, jj_lon], popup="<b>J&J Office</b>", icon=folium.Icon(color="darkblue", icon="briefcase")
).add_to(m)


# HVV Stationen Marker hinzufügen
for name, coords in hvv_stations.items():
    folium.Marker(
        coords, 
        popup=f"<b>{name}</b>", 
        icon=folium.Icon(color="lightblue", icon="train", prefix="fa")
    ).add_to(m)

# Synthetic employee locations
for idx, row in df.iterrows():
    
    if row['high_potential']:
        color = "green" # High Potential Deutschlandticket User
    elif row['time_bucket'] == "60+ Min":
        color = "red"   # Abgehängt
    else:
        color = "orange" # Geringes Potenzial
        
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        weight=1,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=f"Time: {row['travel_time_min']}m<br>Adoption: {row['adoption_potential']*100}%"
    ).add_to(m)


m.save("data/map.html")

### 5. Interactive Map Preview

![Map Preview](data/Screenshot_folium_map.png)

*Note: To ensure optimal performance and avoid rendering crashes within the GitHub or Jupyter environment, the fully interactive map with all 2,000 data points has been exported as a standalone file. Please open the `data/map.html` file in your web browser to explore the interactive version, zoom into neighborhoods, and check individual employee scores.*